In [0]:
%run ../utils/adls_auth

In [0]:
# Disable deletion vectors for Synapse compatibility
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
from pyspark.sql.functions import col, sha2, concat_ws, to_date
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit
import uuid


STREAM_SILVER = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_stream_silver"
GOLD_PATH = "abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips_streaming"

dim_date = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_date")
dim_vendor = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_vendor")
batch_id = str(uuid.uuid4())

In [0]:

stream_df = spark.read.format("delta").load(STREAM_SILVER).withColumn("event_date", to_date(col("event_timestamp")))

fact_df = (
    stream_df
    .join(dim_date, stream_df.event_date == dim_date.full_date, "left")
    .join(dim_vendor, stream_df.vendor_id == dim_vendor.vendor_id, "left")
    .select(
        sha2(concat_ws("_", stream_df.vendor_id, stream_df.pickup_location_id, stream_df.event_time), 256).alias("event_key"),
        dim_date.date_key.alias("event_date_key"),
        dim_vendor.vendor_key,
        stream_df.pickup_location_id,
        stream_df.dropoff_location_id,
        stream_df.trip_distance,
        stream_df.fare_amount,
        stream_df.passenger_count,
        stream_df.event_timestamp,
    )    
    .withColumn("_batch_id", lit(batch_id))
    .withColumn("_created_at", current_timestamp())
    .withColumn("_updated_at", current_timestamp())
)

In [0]:
if not DeltaTable.isDeltaTable(spark, GOLD_PATH):
    fact_df.write.format("delta").mode("overwrite").partitionBy("event_date_key").save(GOLD_PATH)
else:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    fact_table = DeltaTable.forPath(spark, GOLD_PATH)
    
    
    update_cols = [c for c in fact_df.columns if c not in ("_created_at", "_batch_id")]
    update_map = {c: col(f"source.{c}") for c in update_cols}
    update_map["_updated_at"] = current_timestamp()
    
    (fact_table.alias("target")
        .merge(fact_df.alias("source"), "target.event_key = source.event_key")
        .whenMatchedUpdate(set=update_map)
        .whenNotMatchedInsertAll()
        .execute())
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")

print(f"fact_trips_streaming built/merged: {fact_df.count()} rows.")